In [ ]:
#read potential.csv

%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import csv
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# read data from csv file
data = pd.read_csv('../MODE/potentials.csv')
data.describe()


In [ ]:
TH0, TH1 = 1.2, 0.5

def load_data(csv_file):
    # Load the data from the CSV file
    data = pd.read_csv(csv_file)
    return data

def plot_potentials_by_event(data, NL0, NL1):
    """
    Creates subplots for potentials of NL0 and NL1 neurons separately across events.
    Includes buttons to toggle visibility for NL0 and NL1 lines while keeping thresholds always visible.
    """
    total_neurons = NL0 + NL1
    color_potential = [
        'rgb(238,180,248)', 'rgb(241,214,145)', 'rgb(217,234,209)',
        'rgb(224,138,122)', 'rgb(138,216,241)', 'rgb(161,102,242)'
    ]
    
    fig = make_subplots(rows=6, cols=2, shared_xaxes=False, vertical_spacing=0.04)
    traces = []

    for EV in range(1, 13):  # Example range of events (adjust as needed)
        # Decide row and col for the subplot
        row = (EV + 1) // 2
        col = 1 if EV % 2 != 0 else 2

        # Add traces for NL0 neurons
        for neuron in range(NL0):
            scatter = go.Scatter(
                x=data['Time'][data['Event'] == EV],
                y=data[f'V(t)_{neuron}'][data['Event'] == EV],
                mode='lines',
                name=f"L0 N{neuron} Ev{EV} ",
                line=dict(color=color_potential[neuron % len(color_potential)]),
                opacity=0.7,
                visible=True  # Default visibility
            )
            fig.add_trace(scatter, row=row, col=col)
            traces.append((f"L0", scatter))

        # Add traces for NL1 neurons
        for neuron in range(NL0, total_neurons):
            scatter = go.Scatter(
                x=data['Time'][data['Event'] == EV],
                y=data[f'V(t)_{neuron}'][data['Event'] == EV],
                mode='lines',
                name=f"L1 N{neuron - NL0} Ev{EV}",
                line=dict(color=color_potential[neuron % len(color_potential)]),
                opacity=0.7,
                visible=True  # Default visibility
            )
            fig.add_trace(scatter, row=row, col=col)
            traces.append((f"L1", scatter))

        # Add threshold lines (always visible)
        for thresh, label in zip([TH0, TH1], ["TH0", "TH1"]):
            threshold_line = go.Scatter(
                x=[data['Time'][data['Event'] == EV].min(), data['Time'][data['Event'] == EV].max()],
                y=[thresh, thresh],
                mode='lines',
                name=f"Threshold {label}",
                line=dict(color='red', dash='dash'),
                opacity=0.7,
                visible=True  # Always visible
            )
            fig.add_trace(threshold_line, row=row, col=col)
            traces.append((label, threshold_line))

    # Add buttons for toggling L0 and L1 visibility
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                x=0.5,
                y=1.1,
                buttons=[
                    dict(
                        label="Show All L0",
                        method="update",
                        args=[
                            {"visible": [t[0] == "L0" or t[0] == "TH0" for t in traces]},
                            {"title": "Showing All L0 Neurons"}
                        ]
                    ),
                    dict(
                        label="Hide All L0",
                        method="update",
                        args=[
                            {"visible": [t[0] != "L0" or t[0] != "TH0" for t in traces]},
                            {"title": "Hiding All L0 Neurons"}
                        ]
                    ),
                    dict(
                        label="Show All L1",
                        method="update",
                        args=[
                            {"visible": [t[0] == "L1" or t[0] == "TH1" for t in traces]},
                            {"title": "Showing All L1 Neurons"}
                        ]
                    ),
                    dict(
                        label="Hide All L1",
                        method="update",
                        args=[
                            {"visible": [t[0] != "L1" or t[0] != "TH1" for t in traces]},
                            {"title": "Hiding All L1 Neurons"}
                        ]
                    )
                ]
            )
        ]
    )

    fig.update_layout(
        height=800, width=900,
        title_text="Potentials for NL0 and NL1 Neurons by Event",
    )
    fig.show()

# Main execution
if __name__ == "__main__":
    # Load the data
    csv_file = '../MODE/potentials.csv'  # Replace with the actual file path
    data = load_data(csv_file)

    # Define the number of neurons in NL0 and NL1
    NL0 = 10
    NL1 = 10

    # Plot potentials by event
    plot_potentials_by_event(data, NL0, NL1)
